# Minimal MD — the whole pipeline in one place

The bare-bones end-to-end workflow, **self-contained** (no imports beyond the MD stack): fetch → repair → solvate → minimize → simulate → analyze. This is the same logic as `mdtutorial.py`, written flat so you can read every step in one notebook. For the polished figures, see the `fig*` notebooks.

In [ ]:
# make sure the MD stack is importable in THIS kernel (Colab installs it; wrong kernel -> clear message)
import importlib.util, sys, subprocess
_missing = [m for m in ("openmm", "pdbfixer", "mdtraj") if importlib.util.find_spec(m) is None]
if _missing and "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openmm", "pdbfixer", "mdtraj"], check=False)
    _missing = [m for m in _missing if importlib.util.find_spec(m) is None]
if _missing:
    raise SystemExit(f"Missing in this kernel: {_missing}. Select your MD-tutorial conda kernel "
                     "(Kernel > Change Kernel). If it isn't built yet: conda env create -f environment.yml; "
                     "if it exists but is stale: conda env update -f environment.yml.")

In [ ]:
import os, random, urllib.request
import numpy as np, matplotlib.pyplot as plt
import openmm as mm
from openmm import app, unit
from pdbfixer import PDBFixer
import mdtraj as md

PDB_ID    = "1L2Y"                                     # Trp-cage TC5b
TEMP      = 300 * unit.kelvin
SEED      = 2024
N_PROD_PS = int(os.environ.get("N_PROD_PS", "200"))   # production length (ps)

### Fetch and repair
PDBFixer completes the structure; we rebuild hydrogens with the force field. RNGs are seeded and hydrogen placement runs on the **Reference** platform, so preparation is deterministic. (`removeHeterogens` goes *before* `findMissingAtoms` so terminal atoms like OXT still get added.)

In [ ]:
if not os.path.exists(f"{PDB_ID}.pdb"):
    urllib.request.urlretrieve(f"https://files.rcsb.org/download/{PDB_ID}.pdb", f"{PDB_ID}.pdb")
ff = app.ForceField("amber14-all.xml", "amber14/tip3p.xml")
fx = PDBFixer(filename=f"{PDB_ID}.pdb")
fx.findMissingResidues(); fx.findNonstandardResidues()
fx.removeHeterogens(keepWater=False)
fx.findMissingAtoms(); fx.addMissingAtoms()
modeller = app.Modeller(fx.topology, fx.positions)
modeller.delete([a for a in modeller.topology.atoms() if a.element == app.element.hydrogen])
random.seed(SEED); np.random.seed(SEED)
modeller.addHydrogens(ff, pH=7.0, platform=mm.Platform.getPlatformByName("Reference"))
print("repaired atoms:", modeller.topology.getNumAtoms())

### Solvate
Add a periodic TIP3P water box + neutralizing ions. We save the solvated structure so we can load the trajectory against it later.

In [ ]:
random.seed(SEED); np.random.seed(SEED)
modeller.addSolvent(ff, model="tip3p", padding=1.0 * unit.nanometer, neutralize=True)
app.PDBFile.writeFile(modeller.topology, modeller.positions, open("solvated.pdb", "w"))
print("solvated atoms:", modeller.topology.getNumAtoms())

### Build the System, pick a working platform, and minimize
We try platforms in preference order (CUDA → OpenCL → CPU → Reference) and fall back if one is registered but has no usable device. Minimization is a downhill walk on the energy surface — not dynamics.

In [ ]:
system = ff.createSystem(modeller.topology, nonbondedMethod=app.PME,
                         nonbondedCutoff=1.0 * unit.nanometer, constraints=app.HBonds)
names = [mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())]
plat = None
for pname in ("CUDA", "OpenCL", "CPU", "Reference"):
    if pname not in names:
        continue
    try:
        props = {"Precision": "mixed", "DeterministicForces": "true"} if pname == "CUDA" else {}
        integ = mm.LangevinMiddleIntegrator(TEMP, 1 / unit.picosecond, 0.002 * unit.picoseconds)
        integ.setRandomNumberSeed(SEED)
        sim = app.Simulation(modeller.topology, system, integ, mm.Platform.getPlatformByName(pname), props)
        sim.context.setPositions(modeller.positions); plat = pname; break
    except Exception as e:
        print(f"{pname}: registered but unusable ({str(e).splitlines()[0][:50]}) -> next")
print("platform:", plat)
sim.minimizeEnergy()
min_positions = sim.context.getState(getPositions=True).getPositions()
print("minimized PE:", sim.context.getState(getEnergy=True).getPotentialEnergy())

### Simulate
A **fresh Context** — the integrator seed is applied at Context creation, so the Langevin noise stream is reproducible (re-seeding a reused Context does *not* reset the GPU RNG). 20 ps equilibration, then production to `traj.dcd`.

In [ ]:
integ = mm.LangevinMiddleIntegrator(TEMP, 1 / unit.picosecond, 0.002 * unit.picoseconds)
integ.setRandomNumberSeed(SEED)
sim = app.Simulation(modeller.topology, system, integ, mm.Platform.getPlatformByName(plat),
                     {"Precision": "mixed", "DeterministicForces": "true"} if plat == "CUDA" else {})
sim.context.setPositions(min_positions); sim.context.setVelocitiesToTemperature(TEMP, SEED)
sim.step(10000)                                            # 20 ps NVT equilibration
sim.reporters.append(app.DCDReporter("traj.dcd", 500))
sim.step(N_PROD_PS * 500)                                  # production (1 frame / ps)
sim.reporters.clear(); print(f"done: {N_PROD_PS} ps")

### Analyze
Load the trajectory, slice to the protein, superpose onto the minimized start, and plot RMSD over time. *(This RMSD is over **all** protein atoms — one of the definition choices worth being explicit about.)*

In [ ]:
t = md.load("traj.dcd", top="solvated.pdb")
t = t.atom_slice(t.topology.select("protein")); t.superpose(t, 0)
rmsd = md.rmsd(t, t, 0) * 10                                # nm -> Å, all protein atoms
plt.figure(figsize=(7, 4))
plt.plot(np.arange(t.n_frames), rmsd, color="navy", lw=1)
plt.xlabel("time (ps)"); plt.ylabel("RMSD to minimized start (Å)")
plt.title(f"{PDB_ID}: protein RMSD over {N_PROD_PS} ps"); plt.tight_layout(); plt.show()
print(f"mean RMSD {rmsd.mean():.2f} Å, final {rmsd[-1]:.2f} Å")